# PCD Wire Segmentation (SAM3)

**This notebook (sam3 env):**
1. Load `.pcd` → project to synthetic RGBD
2. SAM3 text-prompt segmentation → combined wire mask
3. Back-project mask → filtered `wire_segmented.pcd` + masked RGBD

**Next step (anygrasp env):**
```bash
/home/g20/miniconda3/envs/anygrasp/bin/python \
  /home/g20/projects/anygrasp/anygrasp_sdk/grasp_detection/demo_pcd.py \
  --checkpoint_path .../log/checkpoint_detection.tar \
  --pcd_path .../wire_output/wire_segmented.pcd \
  --scale 1.0 --debug
```

In [ ]:
import os
import numpy as np
import open3d as o3d
import matplotlib.pyplot as plt
from PIL import Image

import torch
import sam3
from sam3 import build_sam3_image_model
from sam3.model.sam3_image_processor import Sam3Processor
from sam3.visualization_utils import plot_results

torch.backends.cuda.matmul.allow_tf32 = True
torch.backends.cudnn.allow_tf32 = True
torch.autocast('cuda', dtype=torch.bfloat16).__enter__()

sam3_root = os.path.join(os.path.dirname(sam3.__file__), '..')
print('SAM3 root:', sam3_root)

In [ ]:
# ─── Configuration ────────────────────────────────────────────────────────────
PCD_PATH   = '/home/g20/Downloads/Test_point_cloud_unorganized.pcd'
OUTPUT_DIR = '/home/g20/projects/anygrasp/anygrasp_sdk/grasp_detection/wire_output'

PCD_SCALE  = 1000.0    # mm -> metres
IMG_W, IMG_H = 1280, 1024
BORDER_FRAC  = 0.05

CONFIDENCE_THRESHOLD = 0.35

# Prompts: wire colour description -> RGB for the combined-mask visualisation
WIRE_PROMPTS = {
    'white wire':  [255, 255, 255],
    'blue wire':   [0,   0,   255],
    'red wire':    [255, 0,   0  ],
    'green wire':  [0,   255, 0  ],
    'yellow wire': [255, 255, 0  ],
}

os.makedirs(OUTPUT_DIR, exist_ok=True)
print('Output dir:', OUTPUT_DIR)

## Step 1 – Load PCD and project to synthetic RGBD

In [ ]:
pcd_raw = o3d.io.read_point_cloud(PCD_PATH)
pts  = np.asarray(pcd_raw.points, dtype=np.float64) / PCD_SCALE   # metres
cols = np.asarray(pcd_raw.colors, dtype=np.float32)               # [0,1]

valid = pts[:, 2] > 0
pts, cols = pts[valid], cols[valid]
print(f'Points: {len(pts):,}')
print(f'XYZ min {pts.min(axis=0)}  max {pts.max(axis=0)}')

X, Y, Z = pts[:, 0], pts[:, 1], pts[:, 2]

# Auto-compute focal length so all points fit with a small border
u_norm, v_norm = X / Z, Y / Z
fx = IMG_W * (1 - 2 * BORDER_FRAC) / (u_norm.max() - u_norm.min())
fy = IMG_H * (1 - 2 * BORDER_FRAC) / (v_norm.max() - v_norm.min())
f  = min(fx, fy)
cx = IMG_W / 2.0 - f * (u_norm.min() + u_norm.max()) / 2.0
cy = IMG_H / 2.0 - f * (v_norm.min() + v_norm.max()) / 2.0
print(f'Virtual camera: f={f:.1f}  cx={cx:.1f}  cy={cy:.1f}')

u_px = np.round(f * X / Z + cx).astype(np.int32)
v_px = np.round(f * Y / Z + cy).astype(np.int32)
in_b  = (u_px >= 0) & (u_px < IMG_W) & (v_px >= 0) & (v_px < IMG_H)
u_s, v_s = u_px[in_b], v_px[in_b]
Z_s, c_s  = Z[in_b].astype(np.float32), cols[in_b]

# Z-buffer: write far -> near so nearest point wins
order = np.argsort(-Z_s)
u_s, v_s, Z_s, c_s = u_s[order], v_s[order], Z_s[order], c_s[order]

depth_img = np.zeros((IMG_H, IMG_W), dtype=np.float32)
color_img = np.zeros((IMG_H, IMG_W, 3), dtype=np.float32)
depth_img[v_s, u_s] = Z_s
color_img[v_s, u_s] = c_s
color_uint8 = (color_img * 255).clip(0, 255).astype(np.uint8)

# Save raw RGBD
Image.fromarray(color_uint8).save(os.path.join(OUTPUT_DIR, 'color.png'))
Image.fromarray((depth_img * 1000).astype(np.uint16)).save(os.path.join(OUTPUT_DIR, 'depth.png'))
print('Saved color.png, depth.png')

fig, axes = plt.subplots(1, 2, figsize=(16, 6))
axes[0].imshow(color_uint8);           axes[0].set_title('Projected colour'); axes[0].axis('off')
axes[1].imshow(depth_img, cmap='plasma'); axes[1].set_title('Projected depth'); axes[1].axis('off')
plt.tight_layout(); plt.show()

## Step 2 – Build SAM3 model

In [ ]:
bpe_path = os.path.join(sam3_root, 'assets', 'bpe_simple_vocab_16e6.txt.gz')
model    = build_sam3_image_model(bpe_path=bpe_path)
print('SAM3 model loaded')

## Step 3 – SAM3 segmentation with wire prompts

In [ ]:
processor       = Sam3Processor(model, confidence_threshold=CONFIDENCE_THRESHOLD)
inference_state = processor.set_image(Image.fromarray(color_uint8))

combined_binary = np.zeros((IMG_H, IMG_W), dtype=np.float32)
combined_colour = np.zeros((IMG_H, IMG_W, 3), dtype=np.uint8)

for prompt, rgb in WIRE_PROMPTS.items():
    processor.reset_all_prompts(inference_state)
    inference_state = processor.set_text_prompt(state=inference_state, prompt=prompt)
    raw = inference_state.get('masks')
    if raw is None or len(raw) == 0:
        print(f'  ✗  {prompt}')
        continue
    m = raw.detach().cpu().numpy()
    if m.ndim == 4:
        m = m.squeeze(1)
    mask_2d = np.max(m, axis=0)
    combined_binary = np.maximum(combined_binary, mask_2d)
    for ch in range(3):
        combined_colour[:, :, ch] = np.where(mask_2d > 0, rgb[ch], combined_colour[:, :, ch])
    print(f'  ✓  {prompt}  ({int(mask_2d.sum()):,} px)')

total_px = int(combined_binary.sum())
print(f'\nTotal wire pixels: {total_px:,} / {IMG_H*IMG_W:,}  ({total_px/(IMG_H*IMG_W)*100:.1f}%)')

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(20, 6))
axes[0].imshow(color_uint8);       axes[0].set_title('Input colour'); axes[0].axis('off')
axes[1].imshow(combined_binary, cmap='gray', vmin=0, vmax=1)
axes[1].set_title('Combined wire mask'); axes[1].axis('off')
axes[2].imshow(combined_colour);   axes[2].set_title('Colour-coded mask'); axes[2].axis('off')
plt.tight_layout(); plt.show()

## Step 4 – Apply mask → save masked RGBD + filtered PCD

In [ ]:
if combined_binary.sum() == 0:
    raise RuntimeError('No wire pixels detected. Adjust prompts or confidence threshold.')

mask_bool = combined_binary > 0

# Masked colour + depth images
masked_color = color_uint8.copy();  masked_color[~mask_bool] = 0
masked_depth = depth_img.copy();    masked_depth[~mask_bool] = 0.0
Image.fromarray(masked_color).save(os.path.join(OUTPUT_DIR, 'wire_color.png'))
Image.fromarray((masked_depth * 1000).astype(np.uint16)).save(os.path.join(OUTPUT_DIR, 'wire_depth.png'))

# Back-project all PCD points into image, keep those inside the wire mask
u_all = np.round(f * X / Z + cx).astype(np.int32)
v_all = np.round(f * Y / Z + cy).astype(np.int32)
in_b2 = (u_all >= 0) & (u_all < IMG_W) & (v_all >= 0) & (v_all < IMG_H)
in_m  = np.zeros(len(pts), dtype=bool)
in_m[in_b2] = mask_bool[v_all[in_b2], u_all[in_b2]]

wire_pts  = pts[in_m].astype(np.float64)
wire_cols = cols[in_m].astype(np.float64)
print(f'Wire points: {len(wire_pts):,} / {len(pts):,}')

wire_pcd_out = o3d.geometry.PointCloud()
wire_pcd_out.points = o3d.utility.Vector3dVector(wire_pts)
wire_pcd_out.colors = o3d.utility.Vector3dVector(wire_cols)
wire_pcd_path = os.path.join(OUTPUT_DIR, 'wire_segmented.pcd')
o3d.io.write_point_cloud(wire_pcd_path, wire_pcd_out)

print(f'Saved wire_segmented.pcd  -> {wire_pcd_path}')
print(f'Saved wire_color.png, wire_depth.png')

fig, axes = plt.subplots(1, 2, figsize=(16, 6))
axes[0].imshow(masked_color);              axes[0].set_title('Masked colour'); axes[0].axis('off')
axes[1].imshow(masked_depth, cmap='plasma'); axes[1].set_title('Masked depth'); axes[1].axis('off')
plt.tight_layout(); plt.show()

print('\n── Next step (anygrasp env) ──')
print(f'/home/g20/miniconda3/envs/anygrasp/bin/python '
      f'/home/g20/projects/anygrasp/anygrasp_sdk/grasp_detection/demo_pcd.py '
      f'--checkpoint_path /home/g20/projects/anygrasp/anygrasp_sdk/grasp_detection/log/checkpoint_detection.tar '
      f'--pcd_path {wire_pcd_path} --scale 1.0 --debug')